# Làm sạch dữ liệu thị trường việc làm IT 2024

Nguồn dữ liệu: **một file Excel** `job_postings_monthly.xlsx`.

- **Nhiều sheet** (mỗi tháng một sheet, ví dụ `01-2024` … `12-2024`): đặt `READ_MODE = "all_sheets"`.
- **Một sheet** chứa cả năm: đặt `READ_MODE = "single_sheet"`.

Upload file lên Colab hoặc trỏ đường dẫn Google Drive ở cell cấu hình.

In [ ]:
# Đọc .xlsx cần openpyxl (Colab thường chưa có sẵn)
!pip install openpyxl -q

import ast
import re
from pathlib import Path

import pandas as pd

In [ ]:
# --- Cấu hình file Excel ---
USE_UPLOAD = True  # False nếu file đã nằm trên Drive/Colab

if USE_UPLOAD:
    from google.colab import files

    uploaded = files.upload()  # chọn file .xlsx
    XLSX_FILE = Path(list(uploaded.keys())[0])
else:
    # from google.colab import drive
    # drive.mount("/content/drive")
    XLSX_FILE = Path("/content/job_postings_monthly.xlsx")
    # XLSX_FILE = Path("/content/drive/MyDrive/.../job_postings_monthly.xlsx")

print("File đọc:", XLSX_FILE)

OUTPUT_FILE = Path("/content/cleaned_job_market_2024.csv")

# "all_sheets" = gộp mọi sheet | "single_sheet" = chỉ 1 sheet
READ_MODE = "all_sheets"

# Khi READ_MODE = "single_sheet": tên sheet (None = sheet đầu tiên)
SINGLE_SHEET_NAME = None

# Khi READ_MODE = "all_sheets": chỉ đọc 12 sheet tháng (bỏ sheet tổng hợp/README)
# Để None = đọc tất cả sheet. Hoặc liệt kê tên sheet đúng trong file của bạn:
MONTHLY_SHEETS = None
# MONTHLY_SHEETS = [f"{m:02d}-2024" for m in range(1, 13)]

In [ ]:
# =============================================================================
# Bước 1: Đọc file Excel và gộp thành một DataFrame
# =============================================================================
xl = pd.ExcelFile(XLSX_FILE)
print("Các sheet trong file:", xl.sheet_names)

if READ_MODE == "single_sheet":
    sheet = SINGLE_SHEET_NAME or xl.sheet_names[0]
    df = pd.read_excel(xl, sheet_name=sheet)
    df["posting_month"] = sheet
    print(f"Đọc 1 sheet '{sheet}' — {len(df):,} dòng.")

elif READ_MODE == "all_sheets":
    sheet_names = MONTHLY_SHEETS if MONTHLY_SHEETS else xl.sheet_names
    frames = []
    for name in sheet_names:
        if name not in xl.sheet_names:
            print(f"  Bỏ qua (không có sheet): {name}")
            continue
        df_month = pd.read_excel(xl, sheet_name=name)
        df_month["posting_month"] = name
        frames.append(df_month)
        print(f"  Sheet '{name}': {len(df_month):,} dòng")

    df = pd.concat(frames, ignore_index=True)
    print(f"Tổng sau khi gộp {len(frames)} sheet: {len(df):,} dòng.")
else:
    raise ValueError('READ_MODE phải là "all_sheets" hoặc "single_sheet"')

df.head()

In [ ]:
# =============================================================================
# Bước 2: Lọc missing — chỉ giữ dòng có salary_year_avg
# =============================================================================
rows_before = len(df)

df["salary_year_avg"] = pd.to_numeric(df["salary_year_avg"], errors="coerce")
df = df.dropna(subset=["salary_year_avg"])

print(f"Đã xóa {rows_before - len(df):,} dòng thiếu lương. Còn lại: {len(df):,}")

In [ ]:
# =============================================================================
# Bước 3: Cột experience_level từ job_title (regex)
# =============================================================================
SENIOR_RE = re.compile(r"\b(Senior|Principal|Lead|Manager)\b", re.IGNORECASE)
JUNIOR_RE = re.compile(r"\b(Junior|Intern|Entry)\b", re.IGNORECASE)


def classify_experience(title):
    if pd.isna(title):
        return "Mid-level"
    t = str(title)
    if SENIOR_RE.search(t):
        return "Senior"
    if JUNIOR_RE.search(t):
        return "Junior"
    return "Mid-level"


df["experience_level"] = df["job_title"].apply(classify_experience)
df["experience_level"].value_counts()

In [ ]:
# =============================================================================
# Bước 4: Làm sạch job_skills — chuỗi → list Python
# =============================================================================
def parse_skills(value):
    """Chuyển "['sql', 'python']" thành ['sql', 'python']."""
    if isinstance(value, list):
        return [str(s).strip().lower() for s in value if str(s).strip()]
    if pd.isna(value):
        return []

    text = str(value).strip()
    if not text or text in ("[]", "nan"):
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(s).strip().lower() for s in parsed if str(s).strip()]
    except (ValueError, SyntaxError):
        pass

    # Fallback khi format lạ
    inner = text.strip("[]").replace("'", "").replace('"', "")
    return [p.strip().lower() for p in inner.split(",") if p.strip()]


df["job_skills"] = df["job_skills"].apply(parse_skills)
print("Ví dụ sau làm sạch:", df["job_skills"].iloc[0])

In [ ]:
# =============================================================================
# Bước 5: Xuất cleaned_job_market_2024.csv
# =============================================================================
df_export = df.copy()
# CSV cần chuỗi — dùng repr(list) để đọc lại bằng ast.literal_eval
df_export["job_skills"] = df_export["job_skills"].apply(repr)

df_export.to_csv(OUTPUT_FILE, index=False)
print(f"Đã lưu: {OUTPUT_FILE}")

# Tải file về máy (Colab)
from google.colab import files

files.download(str(OUTPUT_FILE))